In [2]:
import numpy as np
import pandas as pd

# importation fichier .parquet
httrain = pd.read_parquet("data/history_toulouse_train.parquet")
httest = pd.read_parquet("data/history_toulouse_test.parquet")
httest

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00002 - LAFAYETTE,2026-07-31 23:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - POMME,2025-01-05 05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# ajout serie(colonne) heure en format datetime
t = httrain.index.get_level_values("commit_at")
httrain["heure"] = t.hour
httrain["minute"] = t.minute

httrain

bikes  stands  heure  minute
station                  commit_at                                        
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9     12      15
                         2023-08-05 12:30:00     10       9     12      30
                         2023-08-05 12:45:00     10       9     12      45
                         2023-08-05 13:00:00     11       8     13       0
                         2023-08-05 13:15:00     11       8     13      15
...                                             ...     ...    ...     ...
TEST BEE                 2026-02-12 15:00:00      0       3     15       0
                         2026-02-12 15:15:00      0       3     15      15
                         2026-02-12 15:30:00      0       3     15      30
                         2026-02-12 15:45:00      0       3     15      45
                         2026-02-12 16:00:00      0       3     16       0

[40012622 rows x 4 columns]

In [4]:
# calcul moyenne des velo par quart d'heure
profil_moyen = httrain.groupby(["station", "heure", "minute"])["bikes"].mean()


profil_moyen

station                   heure  minute
00001 - POIDS DE L'HUILE  0      0         2.396884
                                 15        2.318057
                                 30        2.211732
                                 45        2.153987
                          1      0         2.055912
                                             ...   
TEST BEE                  22     45        1.000000
                          23     0         1.000000
                                 15        1.000000
                                 30        1.000000
                                 45        1.000000
Name: bikes, Length: 76528, dtype: float64

In [5]:
horizons = [15, 30, 45, 60, 75, 90, 105, 120]  # adapte à tes colonnes


def moyenne_by_profil_2(prediction_o):
    prediction = prediction_o.copy()
    stations = prediction.index.get_level_values("station")
    commit_at = prediction.index.get_level_values("commit_at")

    for h in horizons:
        t = commit_at + pd.Timedelta(minutes=h)  # moment à prédire

        cle = pd.MultiIndex.from_arrays(
            [stations, t.hour, t.minute],
            names=["station", "heure", "minute"],  # mêmes noms que profil_moyen
        )

        prediction[f"bikes_+{h}min"] = profil_moyen.reindex(cle).to_numpy()
    return prediction

In [6]:
# Creation de l'ensemble de validation
# On récupère les 9 dernières mesures :
# - la 1ère = dernière mesure dans l'ensemble d'apprentissage
# - les 8 suivantes = valeurs à prédire
last_9 = httrain.groupby(level="station").tail(9).copy()
last_9

bikes  stands  heure  minute
station                  commit_at                                        
00001 - POIDS DE L'HUILE 2026-07-31 19:45:00      6      10     19      45
                         2026-07-31 20:00:00      1      15     20       0
                         2026-07-31 20:15:00      1      15     20      15
                         2026-07-31 20:30:00      1      15     20      30
                         2026-07-31 20:45:00      1      15     20      45
...                                             ...     ...    ...     ...
TEST BEE                 2026-02-12 15:00:00      0       3     15       0
                         2026-02-12 15:15:00      0       3     15      15
                         2026-02-12 15:30:00      0       3     15      30
                         2026-02-12 15:45:00      0       3     15      45
                         2026-02-12 16:00:00      0       3     16       0

[7189 rows x 4 columns]

In [7]:
# On remet l'index sous forme de colonnes
last_9 = last_9.reset_index()
last_9

,station,commit_at,bikes,stands,heure,minute
0,00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,6,10,19,45
1,00001 - POIDS DE L'HUILE,2026-07-31 20:00:00,1,15,20,0
2,00001 - POIDS DE L'HUILE,2026-07-31 20:15:00,1,15,20,15
3,00001 - POIDS DE L'HUILE,2026-07-31 20:30:00,1,15,20,30
4,00001 - POIDS DE L'HUILE,2026-07-31 20:45:00,1,15,20,45
...,...,...,...,...,...,...
7184,TEST BEE,2026-02-12 15:00:00,0,3,15,0
7185,TEST BEE,2026-02-12 15:15:00,0,3,15,15
7186,TEST BEE,2026-02-12 15:30:00,0,3,15,30
7187,TEST BEE,2026-02-12 15:45:00,0,3,15,45


In [8]:
last_9["horizon"] = last_9.groupby("station").cumcount()

In [9]:
# Recherche de valeurs manquantes
missing = last_9.loc[
    last_9[["bikes", "stands"]].isna().any(axis=1),
    ["station", "commit_at", "bikes", "stands"],
]

print(missing)

Empty DataFrame
Columns: [station, commit_at, bikes, stands]
Index: []


In [10]:
# On crée une ligne par station et on récupère les 8 valeurs futures
history_toulouse_validation = (
    last_9.pivot(index="station", columns="horizon", values="bikes")
    .drop(columns=0)
    .rename(
        columns={
            1: "bikes_+15min",
            2: "bikes_+30min",
            3: "bikes_+45min",
            4: "bikes_+60min",
            5: "bikes_+75min",
            6: "bikes_+90min",
            7: "bikes_+105min",
            8: "bikes_+120min",
        }
    )
)

# Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station
cutoff_times = last_9[last_9["horizon"] == 0].set_index("station")["commit_at"]

# Ajouter commit_at
history_toulouse_validation["commit_at"] = cutoff_times

# Mettre station + commit_at en MultiIndex
history_toulouse_validation = (
    history_toulouse_validation.reset_index()
    .set_index(["station", "commit_at"])
    .sort_index()
)

history_toulouse_validation

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 19:45:00,1.0,1.0,1.0,1.0,1.0,1.0,9.0,9.0
00002 - LAFAYETTE,2026-07-31 21:45:00,5.0,5.0,5.0,9.0,9.0,9.0,9.0,9.0
00003 - ALSACE - LORRAINE - POMME,2025-11-10 09:45:00,9.0,12.0,15.0,16.0,17.0,17.0,17.0,18.0
00003 - ALSACE-LORRAINE - POMME,2026-07-31 19:45:00,0.0,0.0,0.0,0.0,0.0,0.0,12.0,12.0
00003 - POMME,2025-01-05 03:45:00,17.0,17.0,17.0,17.0,17.0,17.0,17.0,17.0
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 17:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PASTEUR - LAVOISIER,2025-08-05 08:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PLACE GEORGES BRASSENS,2025-08-05 09:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [11]:
# Creation des données à prédire
history_toulouse_basique = history_toulouse_validation.copy()

In [12]:
history_toulouse_basique = moyenne_by_profil_2(history_toulouse_basique)
print(history_toulouse_basique.isna)

<bound method DataFrame.isna of horizon                                                bikes_+15min  \
station                           commit_at                           
00001 - POIDS DE L'HUILE          2026-07-31 19:45:00      4.600733   
00002 - LAFAYETTE                 2026-07-31 21:45:00      7.653846   
00003 - ALSACE - LORRAINE - POMME 2025-11-10 09:45:00      8.928191   
00003 - ALSACE-LORRAINE - POMME   2026-07-31 19:45:00      3.617424   
00003 - POMME                     2025-01-05 03:45:00      7.250482   
...                                                             ...   
NARBONNE - CLOTASSES              2025-08-22 17:15:00      0.000000   
PASTEUR - LAVOISIER               2025-08-05 08:15:00      0.000000   
PLACE GEORGES BRASSENS            2025-08-05 09:15:00      0.000000   
SERVANTY AIRBUS TRAMWAY           2025-08-01 10:15:00      0.000000   
TEST BEE                          2026-02-12 14:00:00      0.500000   

horizon                                     

In [13]:
# Calcul de l'erreur RMSE sur l'ensemble de validation
rmse = np.sqrt(
    np.nanmean(
        (
            history_toulouse_basique.values
            - history_toulouse_validation.loc[history_toulouse_basique.index].values
        )
        ** 2
    )
)

print(f"RMSE : {rmse:.6f}")

RMSE : 5.310663


In [15]:
history_toulouse_basique.to_parquet(
    path="prevision/prediction_S-H-M.parquet", engine="auto", index=False
)